# 04 — RAG and the Cove support interface
Run locally in VS Code using `.venv-rag`. No additional model training or GPU is needed. This notebook documents the actual local app in `app/` and `src/rag.py`.

RAG = retrieve relevant support examples, augment a prompt with them, generate an answer through Groq. Our sources are Bitext examples, not live customer records or verified store policies. The original trained DistilBERT emotion classifier is integrated; the newer experiment is unused.

Read RAG_GUIDE.md for setup. Never place an API key in this notebook. The local app reads the Git-ignored `.env` file; the UI can accept a replacement for the current session.


## Current integration
Language → English search preparation → intent and emotion → retrieval → Groq draft → claim review and citation validation → UI. Emotion uses original English text or a separate faithful translation, not the search rewrite. Uncertain predictions use neutral wording; complaints and confident negative messages receive a local attention flag. No transfer or store action occurs.

See PDF_ALIGNMENT.md for the dataset deviation, measured scores, routing choices and remaining limitations. The following notebook uses existing model artifacts and recorded live responses; it does not train models.


In [1]:
from pathlib import Path
import sys,json
from importlib.metadata import version
# Locate the repository from either its root or the notebooks directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src/rag.py').exists()), Path.cwd())
if not (ROOT/'src/rag.py').exists():
    raise RuntimeError('Place this notebook inside the ITI ChatBot project and select its .venv-rag kernel.')
sys.path.insert(0,str(ROOT))
print('Project:',ROOT)
# print({p:version(p) for p in ['torch','sentence-transformers','faiss-cpu','groq']})


Project: <project-root>


## 1. Knowledge preparation
The pinned Bitext CSV is joined to the saved intent training split. Only training questions become knowledge documents. Each document keeps an instruction, response, intent and stable ID. Validation and test examples are not indexed.

We use a pretrained MiniLM sentence transformer without fine-tuning. Normalized 384-dimensional vectors are stored in a local FAISS IndexFlatIP. Identical response text is deduplicated within each retrieval result. The build is cached; force rebuilding only if the corpus or embedding settings change.


In [2]:
from scripts.build_rag_index import build_index
metadata=build_index()
print(json.dumps(metadata,indent=2))


{
  "status": "built",
  "documents": 19430,
  "dimensions": 384,
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_revision": "c9745ed1d9f207416be6d2e6f8de32d1f16199bf",
  "dataset_url": "https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/resolve/fb86d1b60038970d79fefbfa0c28b47c22d8b961/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv",
  "dataset_sha256": "6f81102b0100b97b8468eb04368033a23206bf1fde9d53500d5806ec1001a434",
  "index": "FAISS IndexFlatIP, normalized vectors",
  "corpus_partition": "intent training split only",
  "build_seconds": 238.09502188899933,
  "checksums": {
    "index.faiss": "21da5b6b6aaf02a99018274f04ccb3c83ddc2c3579abfc00eba190ada8fdaa96",
    "documents.json": "195517b22a24f63331ab6734b279396e0415f3b4d5453c16c9d09abf03f050a3"
  }
}


## 2. Search before generation
Inspect the question, response and cosine similarity before adding an LLM. Similarity is not a calibrated probability of answer correctness. The first search may be slower while the embedding model loads.


In [3]:
from src.rag import SupportRAG
import pandas as pd
rag=SupportRAG()
hits=rag.retrieve('How can I get a copy of my invoice?')
display(pd.DataFrame([{'question':h['instruction'],'intent':h['intent'],
                       'similarity':h['score'],'reference_excerpt':h['response'][:350]} for h in hits]))


<project-root>/.venv-rag/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


,question,intent,similarity,reference_excerpt
0,how can i get invoice #37777,get_invoice,0.840523,Sure! I completely understand your need to acc...
1,how do i get my invoice #85632,get_invoice,0.840066,I'll make it happen! I understand your need to...
2,where could i find my invoice #00108,check_invoice,0.830879,"Rest assured, I'm here to assist you in locati..."
3,how to get invoice #00108?,get_invoice,0.824277,Positively! I understand that you're seeking g...
4,where do I get invoice #00108?,get_invoice,0.821617,I see what you mean your confusion about where...


## 3. Retrieval evaluation
Use a balanced sample of five questions per intent from validation and test. Count whether the first retrieved example has the same intent and whether any of the top five does. These are topic-relevance proxies, not generation accuracy. Generated paraphrases can remain between data partitions.

This cell makes no Groq calls. The test partition is reported for the fixed initial retrieval recipe; use validation for further tuning and seek fresh confirmation after inspecting test failures.


In [4]:
from scripts.evaluate_rag import evaluate_retrieval
retrieval_summary=evaluate_retrieval(rag)
display(retrieval_summary)


Retrieval sample completed: validation 135


Retrieval sample completed: test 135


,partition,examples,top1_same_intent,hit_at_5_same_intent
0,test,135,0.985185,1.0
1,validation,135,0.992593,1.0


## 4. Generate one grounded answer
The live HTTP evaluation sends the question and retrieved examples to Groq. This cell displays its saved result; the commented example shows how to make a fresh call. A JSON schema constrains answer format and source IDs. The application reviews drafts for unsupported claims and action promises, validates citations, and falls back honestly when validation fails. Human review remains necessary.

The generation prompt rejects instructions embedded in references, does not fill placeholders, and forbids claims that transactions or human handoffs were performed. It asks for clarification when there are multiple requests. These mitigations are tested, not assumed infallible.


In [5]:
# Display the real saved HTTP check without spending API quota again.
# To generate a fresh answer instead: load_dotenv(ROOT/'.env'); answer=rag.answer('How can I track my order?')
from dotenv import load_dotenv
recorded=json.loads((ROOT/'reports/rag/generation_checks.json').read_text())
answer=next(r for r in recorded if r['scenario']=='tracking')
if 'error' in answer: raise RuntimeError(answer['error'])
print(answer['answer'])
display(pd.DataFrame(answer['sources']))


To track your order, go to the store’s website and find the "Track Order" section. There you can enter your order number and see the current status and estimated delivery time.


,id,title,excerpt,topic,similarity
0,4a21d101b03e0e22,"I want to track purchase {{Order Number}}, I n...",Thank you for reaching out! I completely under...,track order,0.811
1,91807140b53ab417,i have got to track purchase {{Order Number}} ...,Thanks for getting in touch! I grasp that you ...,track order,0.808
2,24421b8e1a66050d,i dont know what to do to track purchase {{Ord...,I appreciate you reaching out for assistance w...,track order,0.806
3,30189ed0f9eef0f5,"I need to track purchase {{Order Number}}, can...",Always good to connect! I'm attuned to the fac...,track order,0.804


## 5. Follow-up and multilingual handling
Non-English messages are translated into an English search query; generation replies in the requested language. Follow-up messages are rewritten using the recent conversation. Language detection, translation, retrieval and generation can each introduce errors. Emotion uses the original English wording or a separate faithful translation. Predictions below the saved 0.9 threshold use neutral wording; confident negatives receive priority.


In [6]:
# This is the recorded live follow-up check, not a new generation.
followup=next(r for r in recorded if r['scenario']=='followup')
if 'error' in followup: raise RuntimeError(followup['error'])
print('History supplied:',followup['history'])
print(followup['answer'])
# For a fresh conversation use rag.answer(message, history).


History supplied: [{'role': 'user', 'content': 'How can I track my order?'}, {'role': 'assistant', 'content': 'To track your order, go to the store’s website and find the "Track Order" section. There you can enter your order number and see the current status and estimated delivery time.'}]
If you can’t locate the tracking information on the store’s website, try the help or support section of the site. There you’ll usually find a search bar where you can type in “track order” or “order status.” If the page still doesn’t show what you need, you can reach out to the store’s customer support through the contact form or chat option on the website for further assistance.


## 6. Development checks for answer behavior
The HTTP checks include tracking, refunds, unrelated content, multiple requests, human support, unknown policy, Arabic, prompt override, complaints, positive feedback, invoices and follow-up context. Running the following cell runs the scenarios listed in scripts/evaluate_rag.py (generation, review and sometimes translation each use an API call). Outputs are saved for manual inspection. This is not an independent accuracy benchmark.

Review whether each answer is supported, avoids fictitious actions/policies, handles the user's language and selects suitable sources.


In [7]:
# Optional live evaluation. Uncomment to rerun after reading the existing report.
# from scripts.evaluate_rag import evaluate_generation
# results=evaluate_generation(rag)
report_path=ROOT/'reports/rag/generation_checks.json'
if report_path.exists():
    results=json.loads(report_path.read_text())
    display(pd.DataFrame([{'scenario':r['scenario'],'status':r.get('status'),
                           'answer':r.get('answer',r.get('error'))} for r in results]))
else:
    print('No generation evaluation report yet. Run the optional evaluation when ready.')


,scenario,status,answer
0,greeting,conversation,"Hello! How can I help with your order, account..."
1,tracking,answered,"To track your order, go to the store’s website..."
2,refund,answered,"To request a refund, go to the store’s website..."
3,unsupported,not_supported,"I’m sorry, but I can’t help with that. I can o..."
4,multiple_requests,needs_clarification,It looks like you want to cancel an order and ...
5,human,human_requested,I can’t connect you to an agent or send a requ...
6,unknown_policy,not_supported,"I’m sorry, but I don’t have a confirmed refund..."
7,arabic,answered,للمتابعة، قم بزيارة موقع المتجر وسجّل الدخول إ...
8,override_attempt,answered,"I’m sorry, but I can’t confirm that your order..."
9,frustrated_delay,answered,I’m sorry to hear your order has been delayed....


## 7. Use the interface
Open **Start Cove.command** in the project, then visit http://127.0.0.1:8000. Keep the terminal running. The app displays source cards, retains conversation only for the current page session, and supports a new conversation, copied replies and private key replacement.

The server binds only to this computer. Questions and relevant context go to Groq. There is no production authentication or real transaction integration.

## What to explain in assessment
- Why embeddings search meaning while TF-IDF/SVM classifies intent.
- Why normalized inner product equals cosine similarity.
- Why the corpus excludes validation/test questions.
- How source documents enter the generation prompt and how citations are checked.
- Why valid citations and high retrieval scores do not prove answer correctness.
- The limitations of synthetic reference answers and language translation.

Sources: https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html ; https://console.groq.com/docs/structured-outputs ; https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset
